# Day 2 Lab: The Extended Kalman Filter (EKF)
**Student Edition**

---
### 🎯 Learning Objectives
1. Understand why linear Kalman filters fail on nonlinear vehicle kinematics and polar sensor models.
2. Formulate the nonlinear motion and measurement equations for 2D target and vehicle tracking.
3. Compute and implement the four fundamental EKF Jacobians: $\mathbf{F}_{k-1}, \mathbf{L}_{k-1}, \mathbf{H}_k, \mathbf{M}_k$.
4. Build a modular, generic `ExtendedKalmanFilter` class in Python.
5. Define physically-grounded Process Noise (Q) and Measurement Noise (R) covariance matrices.
6. Simulate noisy radar (Range & Bearing) measurements and track a maneuvering vehicle.
7. Evaluate filter estimation consistency using $\pm 3\sigma$ covariance envelopes and the NEES metric via Plotly.


## 1. Environment Setup
Import required libraries for scientific computing, Plotly interactive graphics, and statistical validation.


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import chi2

# Set fixed random seed for reproducible simulations
np.random.seed(42)

print('Environment configured: NumPy, SciPy, and Plotly loaded successfully.')


---
## 2. Theoretical Overview: The 5 Equations of the EKF

Given a nonlinear continuous or discrete system:
$$\mathbf{x}_k = \mathbf{f}(\mathbf{x}_{k-1}, \mathbf{u}_{k-1}, \mathbf{w}_{k-1}), \quad \mathbf{w}_{k-1} \sim \mathcal{N}(\mathbf{0}, \mathbf{Q}_{k-1})$$
$$\mathbf{y}_k = \mathbf{h}(\mathbf{x}_k, \mathbf{v}_k), \quad \mathbf{v}_k \sim \mathcal{N}(\mathbf{0}, \mathbf{R}_k)$$

The EKF linearizes $\mathbf{f}(\cdot)$ around the latest posterior $\hat{\mathbf{x}}_{k-1}$ and $\mathbf{h}(\cdot)$ around the prior $\check{\mathbf{x}}_k$:

$$\begin{array}{lll}
\hline
\textbf{Phase} & \textbf{Operation} & \textbf{Mathematical Formulation} \\
\hline
\text{1. State Prediction} & \text{Propagate Nonlinear State} & \check{\mathbf{x}}_k = \mathbf{f}(\hat{\mathbf{x}}_{k-1}, \mathbf{u}_{k-1}, \mathbf{0}) \\[4pt]
\text{2. Covariance Prediction} & \text{Propagate Uncertainty via Jacobians} & \check{\mathbf{P}}_k = \mathbf{F}_{k-1} \hat{\mathbf{P}}_{k-1} \mathbf{F}_{k-1}^T + \mathbf{L}_{k-1} \mathbf{Q}_{k-1} \mathbf{L}_{k-1}^T \\[4pt]
\text{3. Kalman Gain} & \text{Compute Optimal Gain} & \mathbf{K}_k = \check{\mathbf{P}}_k \mathbf{H}_k^T \left( \mathbf{H}_k \check{\mathbf{P}}_k \mathbf{H}_k^T + \mathbf{M}_k \mathbf{R}_k \mathbf{M}_k^T \right)^{-1} \\[4pt]
\text{4. State Correction} & \text{Update with Measurement Innovation} & \hat{\mathbf{x}}_k = \check{\mathbf{x}}_k + \mathbf{K}_k \left( \mathbf{y}_k - \mathbf{h}(\check{\mathbf{x}}_k, \mathbf{0}) \right) \\[4pt]
\text{5. Covariance Correction} & \text{Update Covariance Matrix} & \hat{\mathbf{P}}_k = (\mathbf{I} - \mathbf{K}_k \mathbf{H}_k) \check{\mathbf{P}}_k \\[4pt]
\hline
\end{array}$$

where the Jacobians are defined as:
$$\mathbf{F}_{k-1} = \left. \frac{\partial \mathbf{f}}{\partial \mathbf{x}} \right|_{\hat{\mathbf{x}}_{k-1}, \mathbf{u}_{k-1}, \mathbf{0}}, \quad
\mathbf{L}_{k-1} = \left. \frac{\partial \mathbf{f}}{\partial \mathbf{w}} \right|_{\hat{\mathbf{x}}_{k-1}, \mathbf{u}_{k-1}, \mathbf{0}}, \quad
\mathbf{H}_k = \left. \frac{\partial \mathbf{h}}{\partial \mathbf{x}} \right|_{\check{\mathbf{x}}_k, \mathbf{0}}, \quad
\mathbf{M}_k = \left. \frac{\partial \mathbf{h}}{\partial \mathbf{v}} \right|_{\check{\mathbf{x}}_k, \mathbf{0}}$$


---
## 3. Implementing the `ExtendedKalmanFilter` Class

### 📝 Exercise 1: Implement the EKF Class


In [ ]:
def wrap_angle(angle):
    """Wraps an angle or array of angles to the interval [-pi, pi]."""
    return (angle + np.pi) % (2 * np.pi) - np.pi

class ExtendedKalmanFilter:
    """Generic Multi-dimensional Extended Kalman Filter (EKF)."""
    
    def __init__(self, x0: np.ndarray, P0: np.ndarray):
        self.x = np.asarray(x0, dtype=np.float64).reshape(-1, 1)
        self.P = np.asarray(P0, dtype=np.float64)
        self.n = self.x.shape[0]
        
    def predict(self, f_func, F_jac: np.ndarray, Q: np.ndarray, 
                u: np.ndarray = None, L_jac: np.ndarray = None):
        # TODO 1.1: Complete state and covariance prediction
        pass  # <-- YOUR CODE HERE
        
    def update(self, y: np.ndarray, h_func, H_jac: np.ndarray, R: np.ndarray, 
               M_jac: np.ndarray = None, angle_indices: list = None):
        # TODO 1.2: Complete measurement update equations
        pass  # <-- YOUR CODE HERE


---
## 4. Vehicle & Sensor Formulation: 2D Polar Radar Tracking

Vehicle State: $\mathbf{x} = [p_x, p_y, v, \theta]^T$, Control: $\mathbf{u} = [a, \omega]^T$.

Nonlinear Kinematics:
$$\mathbf{f}(\mathbf{x}, \mathbf{u}) = \begin{bmatrix} p_x + v \cos(\theta)\Delta t \\ p_y + v \sin(\theta)\Delta t \\ v + a \Delta t \\ \theta + \omega \Delta t \end{bmatrix}$$\n\nPolar Radar Measurement:
$$\mathbf{y} = \mathbf{h}(\mathbf{x}) + \mathbf{v} = \begin{bmatrix} \sqrt{p_x^2 + p_y^2} \\ \operatorname{atan2}(p_y, p_x) \end{bmatrix} + \mathbf{v}$$\n\nMeasurement Jacobian:
$$\mathbf{H} = \begin{bmatrix} \frac{p_x}{r} & \frac{p_y}{r} & 0 & 0 \\ -\frac{p_y}{r^2} & \frac{p_x}{r^2} & 0 & 0 \end{bmatrix}, \quad \text{where } r = \sqrt{p_x^2 + p_y^2}$$


In [ ]:
def motion_model(x, u, dt):
    """Nonlinear kinematic state transition."""
    # TODO 2.1: Implement motion model
    pass

def get_F_jacobian(x, dt):
    """Computes 4x4 state transition Jacobian F."""
    # TODO 2.2: Implement F Jacobian
    pass

def measurement_model(x):
    """Computes range and bearing [r, phi]^T from Cartesian state."""
    # TODO 2.3: Implement measurement model
    pass

def get_H_jacobian(x):
    """Computes 2x4 measurement Jacobian H."""
    # TODO 2.4: Implement H Jacobian
    pass


---
## 5. Physical Formulation of $\mathbf{Q}$ and $\mathbf{R}$ Matrices

### 🔹 Process Noise Covariance $\mathbf{Q}$ ($\mathbf{x} = [p_x, p_y, v, \theta]^T$):
* $\sigma_{p_x} = 0.05\text{ m}, \sigma_{p_y} = 0.05\text{ m}$: Models tire lateral slip and 1st-order discrete kinematic truncation over $\Delta t = 0.1\text{ s}$.
* $\sigma_v = 0.1\text{ m/s}$: Models engine throttle response lag, road slope variations, and braking jitter.
* $\sigma_\theta = 0.02\text{ rad} \approx 1.15^\circ$: Models steering backlash, crosswinds, and road bank angle.
$$\mathbf{Q} = \operatorname{diag}(0.05^2, 0.05^2, 0.1^2, 0.02^2)$$

### 🔹 Measurement Noise Covariance $\mathbf{R}$ ($\mathbf{y} = [r, \phi]^T$):
* $\sigma_r = 0.5\text{ m}$: Automotive mmWave radar range ToF measurement precision.
* $\sigma_\phi = 1.0^\circ = 0.0175\text{ rad}$: Radar antenna array beam azimuth resolution.
$$\mathbf{R} = \operatorname{diag}(0.5^2, (\operatorname{deg2rad}(1.0))^2)$$


---
## 6. Simulation & Filter Execution


In [ ]:
dt = 0.1
T_total = 60.0
N_steps = int(T_total / dt)
time = np.linspace(0, T_total, N_steps)

x_true_all = np.zeros((4, N_steps))
x_true = np.array([10.0, 5.0, 15.0, 0.0]).reshape(-1, 1)

Q = np.diag([0.05**2, 0.05**2, 0.1**2, 0.02**2])
R = np.diag([0.5**2, np.deg2rad(1.0)**2])

measurements = []
for k in range(N_steps):
    t = time[k]
    a_cmd = 0.5 * np.sin(0.1 * t)
    omega_cmd = 0.1 * np.cos(0.08 * t)
    u = np.array([a_cmd, omega_cmd]).reshape(-1, 1)
    w = np.random.multivariate_normal(np.zeros(4), Q).reshape(-1, 1)
    x_true = motion_model(x_true, u, dt) + w
    x_true[3, 0] = wrap_angle(x_true[3, 0])
    x_true_all[:, k] = x_true.flatten()
    v_noise = np.random.multivariate_normal(np.zeros(2), R).reshape(-1, 1)
    y = measurement_model(x_true) + v_noise
    y[1, 0] = wrap_angle(y[1, 0])
    measurements.append((u, y))

x0_est = np.array([8.0, 3.0, 10.0, np.deg2rad(10.0)]).reshape(-1, 1)
P0_est = np.diag([5.0**2, 5.0**2, 5.0**2, np.deg2rad(20.0)**2])
ekf = ExtendedKalmanFilter(x0_est, P0_est)

x_est_all = np.zeros((4, N_steps))
P_diag_all = np.zeros((4, N_steps))
nees_all = np.zeros(N_steps)

for k in range(N_steps):
    u, y = measurements[k]
    F = get_F_jacobian(ekf.x, dt)
    ekf.predict(lambda x, u_in: motion_model(x, u_in, dt), F, Q, u=u)
    H = get_H_jacobian(ekf.x)
    ekf.update(y, measurement_model, H, R, angle_indices=[1])
    x_est_all[:, k] = ekf.x.flatten()
    P_diag_all[:, k] = np.diag(ekf.P)
    err = x_true_all[:, k:k+1] - ekf.x
    err[3, 0] = wrap_angle(err[3, 0])
    nees_all[k] = (err.T @ np.linalg.inv(ekf.P) @ err).item()

print('EKF execution complete. Mean NEES:', np.mean(nees_all))


---
## 7. Plotly Interactive Visualizations

Visualizing:
1. **2D Trajectory:** Ground Truth vs. Radar Measurements vs. EKF Estimate.
2. **Position Error:** Euclidean Position Error with $\pm 3\sigma$ confidence envelope.
3. **Heading Error:** Orientation error in degrees with $\pm 3\sigma$ bounds.
4. **Filter Consistency:** NEES metric compared against 95% $\chi^2$ bounds.


In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '2D Vehicle Trajectory Tracking',
        'Position Error vs. 3-Sigma Bound',
        'Heading Error vs. 3-Sigma Bounds',
        'Filter Consistency: NEES (dim=4)'
    )
)

radar_x = [m[1][0, 0] * np.cos(m[1][1, 0]) for m in measurements]
radar_y = [m[1][0, 0] * np.sin(m[1][1, 0]) for m in measurements]

fig.add_trace(go.Scatter(x=x_true_all[0, :], y=x_true_all[1, :], mode='lines', name='Ground Truth', line=dict(color='black', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=radar_x[::4], y=radar_y[::4], mode='markers', name='Radar Points', marker=dict(color='red', size=4, opacity=0.4)), row=1, col=1)
fig.add_trace(go.Scatter(x=x_est_all[0, :], y=x_est_all[1, :], mode='lines', name='EKF Estimate', line=dict(color='blue', width=2, dash='dash')), row=1, col=1)

pos_err = np.sqrt((x_true_all[0, :] - x_est_all[0, :])**2 + (x_true_all[1, :] - x_est_all[1, :])**2)
sigma_pos = 3.0 * np.sqrt(P_diag_all[0, :] + P_diag_all[1, :])
fig.add_trace(go.Scatter(x=time, y=pos_err, mode='lines', name='Position Error [m]', line=dict(color='blue', width=1.5)), row=1, col=2)
fig.add_trace(go.Scatter(x=time, y=sigma_pos, mode='lines', name='+3-Sigma Bound [m]', line=dict(color='red', dash='dot', width=1.5)), row=1, col=2)

heading_err = wrap_angle(x_true_all[3, :] - x_est_all[3, :])
sigma_heading = 3.0 * np.sqrt(P_diag_all[3, :])
fig.add_trace(go.Scatter(x=time, y=np.rad2deg(heading_err), mode='lines', name='Heading Error [deg]', line=dict(color='green', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=time, y=np.rad2deg(sigma_heading), mode='lines', name='+3-Sigma [deg]', line=dict(color='red', dash='dot', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=time, y=-np.rad2deg(sigma_heading), mode='lines', name='-3-Sigma [deg]', line=dict(color='red', dash='dot', width=1.5), showlegend=False), row=2, col=1)

chi2_low, chi2_high = chi2.interval(0.95, df=4)
fig.add_trace(go.Scatter(x=time, y=nees_all, mode='lines', name='NEES', line=dict(color='black', width=1.2)), row=2, col=2)
fig.add_trace(go.Scatter(x=time, y=[chi2_high]*N_steps, mode='lines', name='95% Upper (9.49)', line=dict(color='red', dash='dash')), row=2, col=2)
fig.add_trace(go.Scatter(x=time, y=[chi2_low]*N_steps, mode='lines', name='95% Lower (0.48)', line=dict(color='orange', dash='dash')), row=2, col=2)

fig.update_layout(title_text='Day 2 EKF: 2D Radar Tracking & Performance Diagnostics', template='plotly_white', height=750, width=1050)
fig.show()


---
## 8. Concept Questions
1. Why is the state prediction step computed using the full nonlinear motion function $\mathbf{f}(\cdot)$ rather than the linear matrix $\mathbf{F}$?
2. What causes NEES to consistently exceed the $95\%$ upper bound if process noise $\mathbf{Q}$ is chosen too small?
